In [32]:
import os

In [33]:
%pwd

'd:\\AI - Courses\\MLOPs\\wine_quality_project'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\AI - Courses\\MLOPs\\wine_quality_project'

In [34]:
import mlflow
from dotenv import load_dotenv

# load from .env
load_dotenv('.env') 

# Set credentials for MLflow with basic auth (required by Dagshub)
os.environ['MLFLOW_TRACKING_USERNAME'] = os.getenv('MLFLOW_TRACKING_USERNAME')
os.environ['MLFLOW_TRACKING_PASSWORD'] = os.getenv('MLFLOW_TRACKING_PASSWORD')
mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI'))

In [35]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_file_name: Path
    all_params: dict
    target_column: str
    dotenv_path: Path

In [36]:
from src.wine_quality_project.constants import *
from src.wine_quality_project.utils.common import read_yaml, create_directories, save_json

In [37]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 schema_fielpath=SCHEMA_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.schema=read_yaml(schema_fielpath)
        self.params=read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self)-> ModelEvaluationConfig:
        config=self.config.model_evaluation
        params=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN

        create_directories([config.root_dir])
        model_evaluation_config=ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path, 
            model_path=config.model_path,
            metric_file_name=config.metric_file_name,
            all_params=params,
            target_column=schema.name,
            dotenv_path=config.dotenv_path
        )
        return model_evaluation_config

In [38]:
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
from mlflow.models import Model
from mlflow.models.signature import infer_signature
import numpy as np
import joblib
import tempfile

In [39]:
class ModelEvaluation:
    def __init__(self,config: ModelEvaluationConfig):
        self.config=config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)
        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        load_dotenv(dotenv_path=self.config.dotenv_path) 
        print(self.config.dotenv_path)
        print(type(self.config.dotenv_path))

        # Set credentials for MLflow with basic auth (required by Dagshub)
        os.environ['MLFLOW_TRACKING_USERNAME'] = os.getenv('MLFLOW_TRACKING_USERNAME')
        os.environ['MLFLOW_TRACKING_PASSWORD'] = os.getenv('MLFLOW_TRACKING_PASSWORD')
        mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI'))

        #mlflow.set_tracking_uri(self.config.mlflow_uri)
        #tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            pred_y = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, pred_y)
            scores = {'rmse': rmse, "mae": mae, 'r2': r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)
            
            mlflow.log_params(self.config.all_params)
            mlflow.log_metric('rmse', rmse)
            mlflow.log_metric('mae', mae)
            mlflow.log_metric('r2', r2)

            singature = infer_signature(test_x, pred_y)

            # # Model registry does not work with file store
            # if tracking_url_type_store != "file":

            #     # Register the model
            #     # There are other ways to use the Model Registry, which depends on the use case,
            #     # please refer to the doc for more information:
            #     # https://mlflow.org/docs/latest/model-registry.html#api-workflow
            #     mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            # else:
            #     mlflow.sklearn.log_model(model, "model")

            # Bypass registry: save model manually and log as artifacts
            with tempfile.TemporaryDirectory() as temp_dir:
                model_path = f"{temp_dir}/model"
                mlflow.sklearn.save_model(sk_model=model, path=model_path, signature=singature)
                mlflow.log_artifacts(model_path, artifact_path='model')

            

In [40]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2025-06-20 15:54:20,067: INFO: common: yaml file: config\config.yaml loaded successfuly]
[2025-06-20 15:54:20,073: INFO: common: yaml file: schema.yaml loaded successfuly]
[2025-06-20 15:54:20,076: INFO: common: yaml file: params.yaml loaded successfuly]
[2025-06-20 15:54:20,078: INFO: common: created directories at: artifacts]
[2025-06-20 15:54:20,080: INFO: common: created directories at: artifacts/model_evaluation]
../.env
<class 'str'>
[2025-06-20 15:54:20,472: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]
🏃 View run indecisive-snipe-590 at: https://dagshub.com/usama44/wine-quality-project.mlflow/#/experiments/0/runs/0c844eb6c8e5441aa8295c7490eaa42c
🧪 View experiment at: https://dagshub.com/usama44/wine-quality-project.mlflow/#/experiments/0
